# Qwen3.5 4B Vision Fine-Tuning for IntersectionQA

Adapted from Unsloth's `Qwen3_5_(4B)_Vision.ipynb` notebook for IntersectionQA public JSONL exports. This notebook trains on the dataset's existing data model: each public row provides a model-facing `prompt`, canonical `answer`, `task_type`, split assignment, labels, diagnostics, and provenance.

IntersectionQA v0.1 is code-only by default. For the vision collator, this notebook always supplies an image field: it uses render contact sheets when you provide them, otherwise it uses a small blank image and the task remains code-only.


## Dataset Input

Expected layout is the normal IntersectionQA export directory:

```text
intersectionqa_v0_1/
  train.jsonl
  validation.jsonl
  test_random.jsonl
  test_generator_heldout.jsonl
  test_object_pair_heldout.jsonl
  test_near_boundary.jsonl
  metadata.json
  source_manifest.json
```

Generate it locally with the repo tooling, then upload or mount it in Colab:

```bash
rtk uv run python -m scripts.generate_dataset --output-dir /tmp/intersectionqa_v0_1
```

Optional render artifacts are expected under `RENDER_DIR/<row_id>/renders/contact_sheet.png`, matching `scripts.render_row_artifacts` output.


## Installation


In [ ]:

%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"
        _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0 datasets pillow
# causal_conv1d is supported only on torch==2.8.0.
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0


## Model


In [ ]:

from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False,  # Use True to reduce VRAM at some quality cost.
    use_gradient_checkpointing = "unsloth",
)


In [ ]:

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## Load IntersectionQA Rows


In [ ]:

from pathlib import Path
import json, random
from collections import Counter
from PIL import Image
from IPython.display import display

DATASET_DIR = Path("/content/intersectionqa_v0_1")
RENDER_DIR = Path("/content/intersectionqa_row_debug")

TRAIN_SPLITS = ["train"]
EVAL_SPLITS = ["validation"]
TASK_TYPES = {"binary_interference", "relation_classification", "volume_bucket"}

MAX_TRAIN_ROWS = 2_000  # Set to None for the full split.
MAX_EVAL_ROWS = 200
RANDOM_SEED = 3407

USE_RENDER_IMAGES = False       # Set True after generating/uploading render artifacts.
REQUIRE_RENDER_IMAGES = False   # Set True to fail fast when a render is missing.
PLACEHOLDER_IMAGE = Image.new("RGB", (32, 32), "white")

PUBLIC_ROW_REQUIRED_KEYS = {
    "id", "dataset_version", "split", "task_type", "prompt", "answer", "script",
    "geometry_ids", "source", "base_object_pair_id", "assembly_group_id", "labels",
    "diagnostics", "difficulty_tags", "label_policy", "hashes", "metadata",
}


def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            missing = PUBLIC_ROW_REQUIRED_KEYS - set(row)
            if missing:
                raise ValueError(f"{path}:{line_number}: missing public-row keys: {sorted(missing)}")
            if row["diagnostics"].get("label_status") != "ok":
                continue
            if row["task_type"] not in TASK_TYPES:
                continue
            yield row


def load_rows(splits, limit=None):
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f"DATASET_DIR does not exist: {DATASET_DIR}. Upload or mount an IntersectionQA export first."
        )
    rows = []
    for split in splits:
        path = DATASET_DIR / f"{split}.jsonl"
        if not path.exists():
            raise FileNotFoundError(f"Missing split file: {path}")
        rows.extend(read_jsonl(path))
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(rows)
    return rows[:limit] if limit else rows

train_rows = load_rows(TRAIN_SPLITS, MAX_TRAIN_ROWS)
eval_rows = load_rows(EVAL_SPLITS, MAX_EVAL_ROWS)

print(f"train rows: {len(train_rows):,}")
print(f"eval rows: {len(eval_rows):,}")
print("train task distribution:", dict(Counter(row["task_type"] for row in train_rows)))
print("train answer distribution:", dict(Counter(row["answer"] for row in train_rows)))


## Convert Rows to Unsloth Vision Conversations


In [ ]:

def prompt_for_model(row):
    return row["prompt"].rstrip() + "\n\nReturn exactly the requested answer and nothing else."


def render_path_for_row(row):
    metadata = row.get("metadata", {}) or {}
    for key in ("image_path", "render_path", "contact_sheet_path"):
        value = metadata.get(key)
        if value and Path(value).exists():
            return Path(value)

    candidates = [
        RENDER_DIR / row["id"] / "renders" / "contact_sheet.png",
        RENDER_DIR / row["id"] / "renders" / "assembly.png",
        DATASET_DIR / "renders" / row["id"] / "contact_sheet.png",
        DATASET_DIR / "renders" / row["id"] / "assembly.png",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def image_for_row(row):
    if USE_RENDER_IMAGES:
        path = render_path_for_row(row)
        if path is not None:
            return Image.open(path).convert("RGB")
        if REQUIRE_RENDER_IMAGES:
            raise FileNotFoundError(f"No render image found for row {row['id']}")
    return PLACEHOLDER_IMAGE.copy()


def convert_to_conversation(row):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_for_row(row)},
                    {"type": "text", "text": prompt_for_model(row)},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ]
    }

converted_train_dataset = [convert_to_conversation(row) for row in train_rows]
converted_eval_dataset = [convert_to_conversation(row) for row in eval_rows]

sample_row = train_rows[0]
print(sample_row["id"], sample_row["split"], sample_row["task_type"], "answer=", sample_row["answer"])
display(image_for_row(sample_row))
print(prompt_for_model(sample_row)[:2_500])


## Baseline Inference Before Fine-Tuning


In [ ]:

from transformers import TextStreamer

FastVisionModel.for_inference(model)


def generate_for_row(row, max_new_tokens=32):
    image = image_for_row(row)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_for_model(row)},
        ],
    }]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.1,
        min_p=0.1,
    )

print("Expected:", eval_rows[0]["answer"])
generate_for_row(eval_rows[0])


## Train


In [ ]:

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 4096,
    ),
)


In [ ]:

trainer_stats = trainer.train()
trainer_stats


## Inference After Fine-Tuning


In [ ]:

FastVisionModel.for_inference(model)

for row in eval_rows[:5]:
    print("\nrow:", row["id"], "task:", row["task_type"], "expected:", row["answer"])
    generate_for_row(row)


## Save LoRA Adapters


In [ ]:

model.save_pretrained("intersectionqa_qwen_lora")
tokenizer.save_pretrained("intersectionqa_qwen_lora")
# model.push_to_hub("YOUR_USERNAME/intersectionqa_qwen_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/intersectionqa_qwen_lora", token="YOUR_HF_TOKEN")


## Optional Merged / GGUF Exports


In [ ]:

# Select only the export you need.

if False:
    model.save_pretrained_merged("intersectionqa_qwen_finetune", tokenizer)

if False:
    model.push_to_hub_merged(
        "YOUR_USERNAME/intersectionqa_qwen_finetune",
        tokenizer,
        token="YOUR_HF_TOKEN",
    )

if False:
    model.save_pretrained_gguf("intersectionqa_qwen_finetune", tokenizer, quantization_method="q8_0")

if False:
    model.push_to_hub_gguf(
        "YOUR_USERNAME/intersectionqa_qwen_finetune",
        tokenizer,
        quantization_method="q8_0",
        token="YOUR_HF_TOKEN",
    )
